## **Estrutura do Dataset**
Para o KNN Item-Based, o foco está na similaridade entre ITENS (marcas/carros), e não entre usuários. A matriz continua sendo Usuário × Item, mas os cálculos de vizinhança são feitos entre colunas (itens).

* **2000 usuários**

* **10 marcas de carro (cada uma com um modelo representativo)**

* **Notas de 0 a 5**

* **~40% de valores ausentes (esparsidade realista)**

## **Geração do Dataset (Python)**

In [1]:
import numpy as np
import pandas as pd

np.random.seed(42)

n_usuarios = 2000

# Cada item (carro) tem um "vetor de características latentes" que define seu perfil
# Dimensões: [preço, esportividade, economia, conforto, off-road]
itens = {
    'Toyota_Corolla':    [0.7, 0.4, 0.8, 0.9, 0.2],
    'VW_Golf':           [0.6, 0.7, 0.6, 0.8, 0.2],
    'Chevrolet_Onix':    [0.4, 0.5, 0.9, 0.5, 0.1],
    'Fiat_Argo':         [0.3, 0.5, 0.9, 0.4, 0.2],
    'Hyundai_HB20':      [0.4, 0.6, 0.8, 0.6, 0.1],
    'Honda_Civic':       [0.8, 0.8, 0.7, 0.9, 0.2],
    'Ford_Ka':           [0.3, 0.6, 0.8, 0.4, 0.1],
    'Renault_Kwid':      [0.2, 0.3, 1.0, 0.3, 0.2],
    'Nissan_Kicks':      [0.5, 0.5, 0.7, 0.7, 0.6],
    'Jeep_Renegade':     [0.6, 0.6, 0.5, 0.7, 0.9],
}
marcas = list(itens.keys())
item_features = np.array(list(itens.values()))  # 10 x 5

# Cada usuário tem um vetor de preferências (mesmas 5 dimensões)
# Usuário avalia um carro conforme alinhamento entre seu gosto e o perfil do carro
n_dims = 5
preferencias = np.random.dirichlet(np.ones(n_dims), size=n_usuarios)  # soma 1

# Nota = similaridade (cosseno) entre preferência e perfil do carro * 5 + ruído
from numpy.linalg import norm
def cosine(a, b):
    return np.dot(a, b) / (norm(a) * norm(b) + 1e-9)

# Normaliza perfis dos itens
item_norm = item_features / (norm(item_features, axis=1, keepdims=True) + 1e-9)

dados = np.zeros((n_usuarios, len(marcas)))
for u in range(n_usuarios):
    for i, marca in enumerate(marcas):
        sim = cosine(preferencias[u], item_norm[i])
        nota = sim * 5 + np.random.normal(0, 0.5)
        dados[u, i] = np.clip(nota, 0, 5)

df = pd.DataFrame(dados, columns=marcas,
                  index=[f'U{i+1:04d}' for i in range(n_usuarios)])

# Introduz esparsidade: ~40% dos usuários não avaliam cada item
mask = np.random.rand(*df.shape) > 0.4
df = df.where(mask).round(1)



In [2]:
print(df.shape)   # (2000, 10)
print(df.head())

(2000, 10)
       Toyota_Corolla  VW_Golf  Chevrolet_Onix  Fiat_Argo  Hyundai_HB20  \
U0001             2.7      3.6             NaN        3.4           4.8   
U0002             NaN      NaN             3.9        NaN           3.6   
U0003             3.3      4.1             NaN        NaN           NaN   
U0004             NaN      5.0             NaN        5.0           4.7   
U0005             NaN      NaN             NaN        3.5           NaN   

       Honda_Civic  Ford_Ka  Renault_Kwid  Nissan_Kicks  Jeep_Renegade  
U0001          4.1      NaN           3.2           NaN            NaN  
U0002          NaN      NaN           NaN           NaN            NaN  
U0003          3.7      3.7           3.4           NaN            NaN  
U0004          4.9      NaN           5.0           NaN            4.2  
U0005          3.5      NaN           NaN           4.8            NaN  


## **Construa um sistema de recomendação baseado em item.**

 * **Dica: construa a matriz de similaridade dos item**